In [36]:
%pip install -qU langchain langchain-ollama langgraph pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import sys
from langchain_core.tools import tool

sys.path.append(os.path.abspath('..'))

caminho_prompt = "../prompts/system_prompt.py"

try:
    with open(caminho_prompt, "r", encoding="utf-8") as arquivo:
        SYSTEM_PROMPT = arquivo.read()
    print("System Prompt carregado")
except FileNotFoundError:
    print(f"Erro: Arquivo não encontrado. Verifique o caminho '{caminho_prompt}'")

# Simulando APIs reais
@tool
def buscar_dados_wearable(patient_id: str) -> str:
    """Busca os últimos dados vitais sincronizados do smartwatch (Apple Health/Google Fit) do paciente. Exemplo de patient_id: 9988"""
    print(f"\n[TOOL EXECUTADA: Buscando dados do wearable para o paciente ID {patient_id}...]")
    return f"Dados do Wearable para {patient_id}: Frequência Cardíaca em repouso 110 bpm. SpO2: 98%."

@tool
def buscar_historico_paciente(patient_id: str) -> str:
    """Busca o histórico médico e receitas de uso contínuo do paciente."""
    print(f"\n[TOOL EXECUTADA: Acessando prontuário do paciente ID {patient_id}...]")
    return f"Prontuário de {patient_id}. Medicações contínuas: Losartana 50mg. Alergias: Nenhuma."

tools = [buscar_dados_wearable, buscar_historico_paciente]

Erro: Arquivo não encontrado. Verifique o caminho '../prompts/system_prompt.md'


In [41]:
import json

caminho_evals = "../evals/sprint1_eval_set.json"

try:
    with open(caminho_evals, "r", encoding="utf-8") as arquivo:
        eval_cases = json.load(arquivo)
    print("Evals carregados")
except FileNotFoundError:
    print(f"Erro: Arquivo não encontrado. Verifique o caminho '{caminho_evals}'")

    for caso in eval_cases:
        id_caso = caso["id"]
        categoria = caso["categoria"]
        entrada = caso["entrada_usuario"]
        esperado = caso["contexto_esperado"]
        
        print(f"[TESTE: {id_caso} | CATEGORIA: {categoria.upper()}]")
        print(f"ENTRADA: {entrada}")
        print(f"ESPERADO: {esperado}")
        
@tool
def buscar_dados_wearable(patient_id: str) -> str:
    """Busca os últimos dados vitais sincronizados do smartwatch (Apple Health/Google Fit) do paciente. Exemplo de patient_id: 9988"""
    print(f"\n[TOOL EXECUTADA: Buscando dados do wearable para o paciente ID {patient_id}...]")
    return f"Dados do Wearable para {patient_id}: Frequência Cardíaca em repouso 110 bpm. SpO2: 98%."

@tool
def buscar_historico_paciente(patient_id: str) -> str:
    """Busca o histórico médico e receitas de uso contínuo do paciente."""
    print(f"\n[TOOL EXECUTADA: Acessando prontuário do paciente ID {patient_id}...]")
    return f"Prontuário de {patient_id}. Medicações contínuas: Losartana 50mg. Alergias: Nenhuma."

tools = [buscar_dados_wearable, buscar_historico_paciente]

Evals carregados


In [42]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage
from prompts.system_prompt import SYSTEM_PROMPT
from langchain_core.tools import tool

# 1. Define your tools here
@tool
def sample_tool(query: str) -> str:
    """A sample tool description."""
    return "Result from tool"

# Create a list of your tools
tools = [sample_tool] 

# 2. Garante memoria da conversa
class State(TypedDict):
    messages: Annotated[list, add_messages]

# 3. Iniciar llama
llm = ChatOllama(model="llama3.1", temperature=0)

# Now it correctly binds a list of tools, not a module
llm_with_tools = llm.bind_tools(tools)

# 4. Chatbot
def chatbot(state: State):
    mensagens_com_sys_prompt = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    resposta = llm_with_tools.invoke(mensagens_com_sys_prompt)
    return {"messages": [resposta]}

# 5. Orquestração LangGraph
graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)

# Pass the list of tools to ToolNode
graph_builder.add_node("tools", ToolNode(tools))

# Rotas
graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition) 
graph_builder.add_edge("tools", "chatbot") 

# 6. Grafo com MemorySaver
memory = MemorySaver()
app = graph_builder.compile(checkpointer=memory)

print("Orquestração LangGraph com memória compilada")

Orquestração LangGraph com memória compilada


In [43]:
def conversar(mensagem, thread_id="sessao_01"):
    print(f"Paciente: {mensagem}")
    
    # thread_id é a chave que diz ao LangGraph para buscar a memória desta conversa específica
    config = {"configurable": {"thread_id": thread_id}}
    
    for evento in app.stream({"messages": [("user", mensagem)]}, config, stream_mode="values"):
        ultima_mensagem = evento["messages"][-1]
        
    print(f"BluaDiagnostics: {ultima_mensagem.content}\n")
    print("-" * 70 + "\n")

print("=== INICIANDO DEMONSTRAÇÃO DO BLUADIAGNOSTICS ===\n")

# 1: forçar o LLM a chamar a tool buscar_dados_wearable
conversar("Meu relógio marcou que minha frequência cardíaca está em 110 bpm em repouso, mas estou me sentindo normal. Meu ID na Care Plus é 9988.")

#  2: testando a memória e continuidade clínica (LLM deve lembrar dos 110 bpm)
conversar("Respondendo à sua pergunta, não tomei café hoje e não passei por estresse. Acabei de acordar.")

# 3: testando as regras de limite de escopo do System Prompt (Out of Scope)
conversar("Entendi. Mudando de assunto, por que o valor do meu plano de saúde aumentou 15% neste mês? Quero contestar a fatura.")

=== INICIANDO DEMONSTRAÇÃO DO BLUADIAGNOSTICS ===

Paciente: Meu relógio marcou que minha frequência cardíaca está em 110 bpm em repouso, mas estou me sentindo normal. Meu ID na Care Plus é 9988.
BluaDiagnostics: "Com base no seu histórico de saúde, sua frequência cardíaca em 110 bpm é considerada alta para alguém em repouso. Isso pode ser um sinal de estresse ou ansiedade, mas também pode indicar problemas mais graves como taquicardia supraventricular (SVT) ou outras condições cardíacas.

*   Você sente palpitações ou batimentos cardíacos irregulares?
*   Além disso, você tem outros sintomas como dor no peito, falta de ar ou fadiga?

----------------------------------------------------------------------

Paciente: Respondendo à sua pergunta, não tomei café hoje e não passei por estresse. Acabei de acordar.
BluaDiagnostics: "É possível que sua frequência cardíaca esteja um pouco alta devido ao acordar. No entanto, é importante monitorar essa condição e verificar se ela persiste.

*   V